In [1]:
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import direction_utils as utils


In [2]:
class EEGNet(nn.Module):
    def __init__(self, nb_classes, Chans=27, Samples=2500, dropoutRate=0.5, 
                 kernLength=64, F1=8, D=2, F2=16, norm_rate=0.25, dropoutType='Dropout'):
        super(EEGNet, self).__init__()
        
        # Handle dropout type
        if dropoutType == 'SpatialDropout2D':
            self.dropout = nn.Dropout2d(dropoutRate)
        elif dropoutType == 'Dropout':
            self.dropout = nn.Dropout(dropoutRate)
        else:
            raise ValueError('dropoutType must be one of SpatialDropout2D or Dropout.')

        # Block 1
        self.conv1 = nn.Conv2d(1, F1, (1, kernLength), padding='same', bias=False)
        self.batchnorm1 = nn.BatchNorm2d(F1)
        self.depthwiseConv = nn.Conv2d(F1, F1*D, (Chans, 1), groups=F1, bias=False)
        self.batchnorm2 = nn.BatchNorm2d(F1*D)
        self.pool1 = nn.AvgPool2d((1, 4))

        # Block 2
        self.separableConv = nn.Conv2d(F1*D, F2, (1, 16), padding='same', bias=False)
        self.batchnorm3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AvgPool2d((1, 8))

        # Flatten and Dense
        self.flatten = nn.Flatten()
        self.dense = nn.Linear(F2 * (Samples // (4 * 8)), nb_classes)
        self.norm_constraint = nn.utils.weight_norm(self.dense)

    def forward(self, x):
        # Block 1
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.depthwiseConv(x)
        x = self.batchnorm2(x)
        x = F.elu(x)
        x = self.pool1(x)
        x = self.dropout(x)

        # Block 2
        x = self.separableConv(x)
        x = self.batchnorm3(x)
        x = F.elu(x)
        x = self.pool2(x)
        x = self.dropout(x)

        # Flatten and Dense
        x = self.flatten(x)
        x = self.dense(x)
        return F.softmax(x, dim=1)

In [3]:
# Function to compute accuracy
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == targets.squeeze()).sum().item()
            total += targets.size(0)
    
    accuracy = correct / total
    return accuracy*100

# Function to calculate accuracy
def calculate_accuracy(preds, labels):
    _, predicted = torch.max(preds, 1)
    correct = (predicted == labels).sum().item()
    accuracy = correct / labels.size(0)
    return accuracy*100

In [4]:
def create_dataset(current_subject, base_path=None):
    """"
    Create training and test dataset for the current subjects.
    
    Parameters:
    current_subject (int): Subject number (1 to 20).
    base_path (str): The directory where .mat files are stored.

    Returns:
    X_train, Y_train, X_test, Y_test
    """
    Xtr_all = []
    Ytr_all = []
    for sub in range(1, current_subject+1):
        rel_path = f'data/S{sub:02d}_mitrials.mat'
        filepath = os.path.join(base_path, rel_path)

        mat_vars = utils.load_mat_file(filepath)
        if len(mat_vars)==2:
            Xtr, Ytr = mat_vars
            Xtr_all.append(Xtr)
            Ytr_all.append(Ytr)

        else:
            Xtr, Ytr, Xte, Yte, Yte_fb = mat_vars
            Xtr_all.append(Xtr)
            Ytr_all.append(Ytr)
            if (sub == current_subject):
                continue
            else:
                Xtr_all.append(Xte)
                Ytr_all.append(Yte)
                # indx = np.where(Yte==Yte_fb)[0]
                # Xtr_all.append(Xte[indx, :, :])
                # Ytr_all.append(Yte[indx])
                
    X_train = np.concatenate(Xtr_all, axis=0)
    Y_train = np.concatenate(Ytr_all, axis=0)

    X_test = Xte
    Y_test = Yte
        
    return X_train, Y_train, X_test, Y_test

In [5]:
def eegnet_model_training(train_loader, val_loader, device, from_scratch=True, verbose=True):
    torch.manual_seed(0)
    # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    learning_rate=1e-3
    num_epochs = 1000
    patience = 30  # Number of epochs to wait before stopping if no improvement
    min_delta = 1e-11 # Minimum change to qualify as improvement

    model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
    if not from_scratch:
        model.load_state_dict(torch.load('calibrated_eegnet_model.pth'))

        # Freeze all layers except the last one
        for param in model.parameters():
            param.requires_grad = False  # Freeze all parameters

        # Unfreeze the last layer parameters
        for param in model.dense.parameters():
            param.requires_grad = True  # Unfreeze last layer

        for param in model.separableConv.parameters():
            param.requires_grad = True

        optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)
    
    else:
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)


    criterion = nn.CrossEntropyLoss().to(device)  # Move loss function to GPU if needed
    # Training with Early Stopping
    best_val_loss = float('inf')
    early_stop_counter = 0

    # Placeholder for training and validation loss history
    train_losses = []
    val_losses = []
    val_accuracy = []

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        # Training Loop
        for inputs, targets in train_loader:  # Assuming train_loader is defined
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets.squeeze())

            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()  # Accumulate loss

        # Calculate average training loss for this epoch
        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # Validation loop (set model to evaluation mode)
        model.eval()
        val_loss = 0.0
        val_acc = 0.0

        with torch.no_grad():  # Disable gradient calculation for validation
            for inputs, targets in val_loader:  # Assuming val_loader is defined
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets.squeeze())
                val_loss += loss.item()

                val_acc += calculate_accuracy(outputs, targets.squeeze())

        # Calculate average validation loss for this epoch
        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        val_acc /= len(val_loader)
        val_accuracy.append(val_acc)
        if verbose:
            print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}%')

        # Early stopping check
        if val_loss < best_val_loss - min_delta:  # Check if validation loss improved
            best_val_loss = val_loss
            early_stop_counter = 0  # Reset early stop counter
            torch.save(model.state_dict(), 'best_model.pth')  # Save best model
        else:
            early_stop_counter += 1

        if early_stop_counter >= patience and epoch > 100:
            print(f'Early stopping at epoch {epoch+1}')
            break

    # Load the best model before returning
    # model.load_state_dict(torch.load('best_model.pth'))
    # print('Training complete.')

    return model

In [6]:
def eegnet_model_evaluation(model, device, test_loader):
    # Validation loop (set model to evaluation mode)
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient calculation for validation
        for inputs, targets in test_loader:  # Assuming val_loader is defined
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)

            # Get predictions and calculate accuracy
            _, predicted = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (predicted == targets.squeeze()).sum().item()

            # test_acc = calculate_accuracy(outputs, targets.squeeze())
    
        # Calculate average loss and accuracy
    test_acc =  100*correct / total

    return test_acc

# torch.manual_seed(0)
# parent_dir = os.path.dirname(os.getcwd())
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# batch_size=32

# model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
# model.load_state_dict(torch.load('best_model.pth'))
# for sub in range(8, 21):
#     Xtr, Ytr, Xte, Yte = utils.online_sess_dataset(sub, base_path=parent_dir)

#     X_test = utils.baseline_correction(Xte)
#     X_test = utils.bandpass_filtering(X_test)

#     X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
#     Y_test_tensor = torch.tensor(Yte, dtype=torch.long).to(device)
#     test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

#     test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

#     test_acc = eegnet_model_evaluation(model, device, test_loader)
    
#     print(f'Subject: {sub:02d}, Test Accuracy: {test_acc:.2f}%')

In [8]:
torch.manual_seed(0)
parent_dir = os.path.dirname(os.getcwd())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size=32
fs = 500
train_ratio = 0.9
sub = 7 #Till subject 7 (starting from 0), only calibration sessions are conducted

# Xtr, Ytr = create_dataset(sub, base_path=parent_dir)
Xtr, Ytr = utils.calib_sess_dataset(base_path=parent_dir)
X_train = utils.baseline_correction(Xtr)
X_train = utils.bandpass_filtering(X_train)

# Creating train-validation split
eeg_train, eeg_val, label_train, label_val = train_test_split(X_train, Ytr, 
                                                              train_size=train_ratio, random_state=42, shuffle=True)


X_train_tensor = torch.tensor(eeg_train, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
Y_train_tensor = torch.tensor(label_train, dtype=torch.long).to(device)  # Use long for classification

X_val_tensor = torch.tensor(eeg_val, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
Y_val_tensor = torch.tensor(label_val, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_val_tensor, Y_val_tensor)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

model = eegnet_model_training(train_loader, val_loader, device, verbose=True)
torch.save(model.state_dict(), 'calibrated_eegnet_model.pth')  # Save best model

TypeError: eegnet_model_training() got an unexpected keyword argument 'verbose'

In [8]:
torch.manual_seed(0)
parent_dir = os.path.dirname(os.getcwd())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size=32

model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
model.load_state_dict(torch.load('calibrated_eegnet_model.pth'))

online_perf = dict()
for sub in range(8, 21):
    Xtr, Ytr, Xte, Yte = utils.online_sess_dataset(sub, base_path=parent_dir)

    X_test = utils.baseline_correction(Xte)
    X_test = utils.bandpass_filtering(X_test)

    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_test_tensor = torch.tensor(Yte, dtype=torch.long).to(device)
    test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

    test_acc = eegnet_model_evaluation(model, device, test_loader)
    online_perf[f'Sub{sub:02d}'] = test_acc

    print(f'Subject: {sub:02d}, Test Accuracy: {test_acc:.2f}%')

perf_df = pd.DataFrame(online_perf)

d:\Praveen\PostDoc@SIT\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:134: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
C:\Users\postd\AppData\Local\Temp\ipykernel_29000\772752469.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `

Subject: 08, Test Accuracy: 45.83%
Subject: 09, Test Accuracy: 50.00%
Subject: 10, Test Accuracy: 52.08%
Subject: 11, Test Accuracy: 41.67%
Subject: 12, Test Accuracy: 52.08%
Subject: 13, Test Accuracy: 58.33%
Subject: 14, Test Accuracy: 62.50%
Subject: 15, Test Accuracy: 41.67%
Subject: 16, Test Accuracy: 50.00%
Subject: 17, Test Accuracy: 58.33%
Subject: 18, Test Accuracy: 50.00%
Subject: 19, Test Accuracy: 50.00%
Subject: 20, Test Accuracy: 72.92%


In [13]:
parent_dir = os.path.dirname(os.getcwd())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Xcalib, Ycalib = utils.calib_sess_dataset(base_path=parent_dir)

train_ratio = 0.9
batch_size=32
perf = dict()
for sub in range(8, 21):
    Xtr, Ytr, Xte, Yte = utils.online_sess_dataset(sub, base_path=parent_dir)
    eeg_train, eeg_val, label_train, label_val = train_test_split(Xtr, Ytr, 
                                                              train_size=train_ratio, random_state=42, shuffle=True)
    
    Xtrain = np.concatenate((Xcalib, eeg_train), axis=0)
    Ytrain = np.concatenate((Ycalib, label_train), axis=0)
    
    # Xtrain = eeg_train
    # Ytrain = label_train

    X_train = utils.baseline_correction(Xtrain)
    X_train = utils.bandpass_filtering(X_train)

    Xval = utils.baseline_correction(eeg_val)
    Xval = utils.bandpass_filtering(Xval)
    

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_train_tensor = torch.tensor(Ytrain, dtype=torch.long).to(device)  # Use long for classification

    X_val_tensor = torch.tensor(Xval, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_val_tensor = torch.tensor(label_val, dtype=torch.long).to(device)

    train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
    test_dataset = TensorDataset(X_val_tensor, Y_val_tensor)

    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

    # model = eegnet_model_training(train_loader, val_loader, device, from_scratch=False)
    model = eegnet_model_training(train_loader, val_loader, device, from_scratch=True)


    # --------------------------------------------------------------------------- #
    X_test = utils.baseline_correction(Xte)
    X_test = utils.bandpass_filtering(X_test)

    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_test_tensor = torch.tensor(Yte, dtype=torch.long).to(device)
    test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

    test_acc = eegnet_model_evaluation(model, device, test_loader)
    perf[f'Sub{sub}'] = test_acc
    
    print(f'Subject: {sub:02d}, Test Accuracy: {test_acc:.2f}%')

    Xcalib = np.concatenate((Xcalib, Xtr, Xte), axis=0)
    Ycalib = np.concatenate((Ycalib, Ytr, Yte), axis=0)

Early stopping at epoch 122


C:\Users\postd\AppData\Local\Temp\ipykernel_39444\827685025.py:96: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))


Subject: 08, Test Accuracy: 52.08%
Early stopping at epoch 102
Subject: 09, Test Accuracy: 39.58%
Early stopping at epoch 102
Subject: 10, Test Accuracy: 54.17%
Early stopping at epoch 102
Subject: 11, Test Accuracy: 56.25%
Early stopping at epoch 102
Subject: 12, Test Accuracy: 54.17%
Early stopping at epoch 121
Subject: 13, Test Accuracy: 56.25%
Early stopping at epoch 102
Subject: 14, Test Accuracy: 52.08%
Early stopping at epoch 102
Subject: 15, Test Accuracy: 54.17%
Early stopping at epoch 102
Subject: 16, Test Accuracy: 56.25%
Early stopping at epoch 102
Subject: 17, Test Accuracy: 41.67%
Early stopping at epoch 102
Subject: 18, Test Accuracy: 58.33%
Early stopping at epoch 212
Subject: 19, Test Accuracy: 56.25%
Early stopping at epoch 102
Subject: 20, Test Accuracy: 54.17%


In [11]:
mean_acc = list(perf.values())
print(mean_acc)
print(f'Average Accuracy: {np.mean(mean_acc)}')

[58.333333333333336, 56.25, 50.0, 52.083333333333336, 50.0, 62.5, 62.5, 54.166666666666664, 50.0, 47.916666666666664, 58.333333333333336, 60.416666666666664, 66.66666666666667]
Average Accuracy: 56.089743589743584
